# ACE-Step instrumentals (Colab T4)

Plain [ACE-Step v1-3.5B](https://github.com/ace-step/ACE-Step) text-to-music. Each track is one generation at the model's maximum of 4 minutes, saved exactly as ACE-Step writes it. There's no extending, stitching, fading, mastering or effects.

It follows how ACE-Step's own [examples](https://ace-step.github.io/) are made:
- **Prompts** are short comma-separated tags: genre, instruments, mood, key, bpm. For example `saxphone, jazz` or `sonata, piano, Violin, B Flat Major, allegro`.
- **Instrumentals** use `[inst]` as the lyrics, or bare structure tags with no words (`[verse]`, `[chorus]`, `[solo]`, `[outro]`).
- **Settings** are ACE-Step's defaults, used by every official example: 60 steps, guidance 15, APG, ERG on.
- **Precision** is bfloat16, the model's native precision.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0),
      "| bf16 supported:", torch.cuda.is_bf16_supported())

## 2. Hear what ACE-Step makes (official demos)

These are the eight instrumentals on ACE-Step's demo page, with the exact prompt and lyrics field each one was made with. Both are fetched live from ace-step.github.io. Use them as a reference for the model's quality and for how to write prompts. You can run this cell before the install.

In [ ]:
import json, urllib.request
from IPython.display import Audio, display

SITE = "https://ace-step.github.io"
CDN = "https://cdn.jsdelivr.net/gh/ace-step/ace-step.github.io/opus/samples"

def fetch_text(url):
    try:
        return urllib.request.urlopen(url, timeout=20).read().decode("utf-8", "replace").strip()
    except Exception as exc:
        return f"(couldn't fetch: {exc})"

samples = json.loads(urllib.request.urlopen(f"{SITE}/samples_data.json", timeout=20).read())
for s in samples["Instrumentals"]:
    raw = f"{SITE}/raw/samples/{s['directory']}/{s['fileName']}"
    print(f"{s['fileName']}\n  prompt: {fetch_text(raw + '_prompt.txt')}\n  lyrics: {fetch_text(raw + '.txt')!r}")
    display(Audio(url=f"{CDN}/{s['directory']}/{s['fileName']}.opus"))

## 3. Install ACE-Step

ACE-Step pins old package versions (e.g. `spacy==3.8.4`) that have no builds for Colab's Python 3.13. So it's installed with `--no-deps`, followed by the packages inference needs. `py3langid` stays at 0.3.0, because 0.4.0 removed a function ACE-Step calls.

pip will print `ERROR: pip's dependency resolver ... ace-step 0.2.0 requires X==...`. That's only a warning about those pins. If the version table prints at the end, the install worked.

In [ ]:
!pip install -q --no-deps git+https://github.com/ace-step/ACE-Step.git
!pip install -q "diffusers>=0.33.0" "spacy>=3.8.7,<3.9" loguru pypinyin "py3langid==0.3.0" hangul-romanize num2words soundfile librosa peft accelerate
import importlib.metadata as md
for pkg in ["ace-step", "torch", "transformers", "diffusers", "py3langid"]:
    print(f"{pkg:13s}", md.version(pkg))

## 4. Hugging Face login (optional)

The weights are public, so a token isn't required, but it speeds up the 8 GB download. Add it as a Colab secret named `HF_TOKEN`: click 🔑 in the left sidebar, then turn on Notebook access.

In [ ]:
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
    print("No HF_TOKEN secret; downloading anonymously.")
if token:
    os.environ["HF_TOKEN"] = token
    login(token=token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")

## 5. Load the model

- **`DTYPE`:**
  - `bfloat16` is the model's native precision and ACE-Step's own default (`--bf16 true`), so it gives the best quality. A T4 has no bfloat16 hardware, so it runs slower.
  - `float16` is faster on a T4, but it isn't what the model was trained in, and it can sound harsher or noisier. That was the setting behind the noisy results before.
- **`OVERLAPPED_DECODE`:** ACE-Step's low-memory decode. Leave it off unless you hit CUDA out-of-memory.
- **First run:** downloads about 8 GB.

In [ ]:
import os, time
import torch
DTYPE = "bfloat16"  #@param ["bfloat16", "float16"]
OVERLAPPED_DECODE = False  #@param {type:"boolean"}
CHECKPOINT_DIR = "/content/ace_step_checkpoints"  #@param {type:"string"}
os.environ["ACE_PIPELINE_DTYPE"] = DTYPE   # ACE-Step reads this and it overrides its default

import soundfile as sf
from acestep.pipeline_ace_step import ACEStepPipeline

# Newer torchaudio routes save() through torchcodec, which Colab doesn't have.
# This writes the same float samples to WAV with soundfile; the audio is unchanged.
def _save_wav_file(self, target_wav, idx, save_path=None, sample_rate=48000, format="wav"):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    sf.write(save_path, target_wav.float().cpu().numpy().T, sample_rate, subtype="FLOAT")
    return save_path
ACEStepPipeline.save_wav_file = _save_wav_file

t0 = time.time()
pipe = ACEStepPipeline(checkpoint_dir=CHECKPOINT_DIR, overlapped_decode=OVERLAPPED_DECODE)
pipe.load_checkpoint(pipe.checkpoint_dir)
pipe.loaded = True
print(f"Loaded in {time.time() - t0:.0f}s | dtype={pipe.dtype} | "
      f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 6. Prompts

Put one prompt per line in `PROMPTS`, in the same tag style as the demos in step 2. Each line becomes one track.

- **Tags that work well:** name instruments directly (`piano`, `rhodes`, `upright bass`, `brushed drums`, `saxophone`, `nylon guitar`, `strings`), then a genre (`jazz`, `bossa nova`, `neo soul`, `ambient`), a mood, and optionally a key and a bpm.
- **Tags that bring noise:** `lo-fi`, `lofi`, `vinyl` and `tape` make ACE-Step reproduce the crackle and hiss from those recordings in its training data. They're left out of the examples below.
- **`LYRICS`:**
  - `[inst]` is what most of the official instrumentals use.
  - `structure` uses bare section tags (`[verse]`, `[chorus]`, `[solo]`, `[outro]`), as in the official `saxphone, jazz` and tango examples. That gives the track sections and a solo.
- **`SEED`:** leave blank to get a random seed per track. The seed is printed, so you can repeat a track you like.

In [ ]:
PROMPTS = """
jazz, piano, upright bass, brushed drums, mellow, relaxing, slow tempo, 75 bpm
chillhop, rhodes, electric piano, jazzy chords, soft drums, bass guitar, warm, 85 bpm
saxophone, jazz, smooth, night, piano, double bass
bossa nova, nylon guitar, soft percussion, flute, warm, 90 bpm
neo soul, rhodes, bass guitar, drums, groovy, laid back, 80 bpm
ambient, piano, strings, calm, peaceful, slow
""".strip().splitlines()
LYRICS = "[inst]"  #@param ["[inst]", "structure"]
DURATION = 240  #@param {type:"slider", min:30, max:240, step:10}
SEED = ""  #@param {type:"string"}
SCHEDULER = "euler"  #@param ["euler", "pingpong", "heun"]
SAVE_TO_DRIVE = True  #@param {type:"boolean"}

LYRICS_TEXT = {"[inst]": "[inst]",
               "structure": "[verse]\n\n[chorus]\n\n[solo]\n\n[verse]\n\n[chorus]\n\n[outro]"}[LYRICS]
print(f"{len(PROMPTS)} track(s) of {DURATION}s, scheduler={SCHEDULER}, lyrics={LYRICS}")

## 7. Generate

Each track is saved as ACE-Step's own 48 kHz float WAV, with a `.txt` next to it recording the prompt and seed. The player under each track uses an MP3 copy, to keep the notebook light; the WAV is the real output. With `SAVE_TO_DRIVE` on, the files go to `MyDrive/ace_step_tracks/`.

- **`euler`** is what ACE-Step's UI recommends and what every official example uses.
- **`pingpong`** is a newer SDE sampler. The README says it improves consistency and style alignment, so try it if a prompt comes out muddy.

In [ ]:
import random, subprocess
from datetime import datetime
from pathlib import Path
from IPython.display import Audio, display

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    out_dir = Path("/content/drive/MyDrive/ace_step_tracks")
else:
    out_dir = Path("/content/ace_step_tracks")
out_dir.mkdir(parents=True, exist_ok=True)

for n, prompt in enumerate(PROMPTS, 1):
    seed = int(SEED) + n - 1 if SEED.strip() else random.SystemRandom().randrange(2 ** 31)
    name = f"{datetime.now():%Y%m%d-%H%M%S}_{n:02d}_seed{seed}"
    t0 = time.time()
    # ACE-Step's defaults, as in every official example
    pipe(format="wav", audio_duration=float(DURATION), prompt=prompt, lyrics=LYRICS_TEXT,
         infer_step=60, guidance_scale=15.0, scheduler_type=SCHEDULER, cfg_type="apg",
         omega_scale=10.0, manual_seeds=[seed], guidance_interval=0.5,
         guidance_interval_decay=0.0, min_guidance_scale=3.0, use_erg_tag=True,
         use_erg_lyric=True, use_erg_diffusion=True, oss_steps=None,
         guidance_scale_text=0.0, guidance_scale_lyric=0.0,
         save_path=str(out_dir / f"{name}.wav"), batch_size=1)
    (out_dir / f"{name}.txt").write_text(
        f"prompt: {prompt}\nlyrics: {LYRICS_TEXT!r}\nseed: {seed}\nscheduler: {SCHEDULER}\n"
        f"dtype: {DTYPE}\nduration: {DURATION}\n")
    print(f"[{n}/{len(PROMPTS)}] {time.time() - t0:.0f}s | seed {seed}\n  {prompt}")
    # Playback only: a 4-minute float WAV is ~90 MB to embed, so the player gets a
    # 256 kbps MP3 copy. The saved WAV is untouched.
    preview = Path("/content/preview.mp3")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(out_dir / f"{name}.wav"),
                    "-b:a", "256k", str(preview)], check=True)
    display(Audio(str(preview)))
print("Saved in", out_dir)